# AF2 complementary mechanisms — static audit
Audit dijalankan dalam proses Python baru agar tidak memakai cache modul Colab. Tidak melakukan training atau membuka test.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import importlib, json, os, shutil, subprocess, sys, torch
from pathlib import Path

assert torch.cuda.is_available(), 'Aktifkan T4 GPU.'
REPO = Path('/content/coffee-bean-detection')
BRANCH = 'codex/af2-complementary-mechanisms'
os.chdir('/content')
if REPO.exists():
    shutil.rmtree(REPO)
clone = ['git', 'clone', '--depth', '1', '--branch', BRANCH, 'https://github.com/ediprin/coffee-bean-detection.git', str(REPO)]
for attempt in range(3):
    completed = subprocess.run(clone)
    if completed.returncode == 0:
        break
    if REPO.exists():
        shutil.rmtree(REPO)
else:
    raise RuntimeError('Git clone gagal tiga kali.')
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'ultralytics==8.4.96', '-e', str(REPO)], check=True)
sys.path.insert(0, str(REPO / 'src'))
importlib.invalidate_caches()
os.chdir(REPO)
print('SETUP SELESAI:', BRANCH)

In [ ]:
from coffee_detector.drive_project import require_project_artifact, resolve_drive_project_root

AF2_REL = 'experiments/faruq-v3-breadth-screening-batch-v1/candidates/AFAB/AF2_seed42/weights/best.pt'
PROJECT = resolve_drive_project_root(required_relative_paths=(AF2_REL,))
AF2 = require_project_artifact(PROJECT, AF2_REL)
OUTPUT = PROJECT / 'experiments/faruq-v3-af2-complement-v1'
STATIC = OUTPUT / 'static_audit.json'
LOG = OUTPUT / 'static_audit_run.log'
OUTPUT.mkdir(parents=True, exist_ok=True)
print('PROJECT:', PROJECT)
print('AF2:', AF2)
print('STATIC:', STATIC)

In [ ]:
command = [
    sys.executable, '-u', '-m',
    'coffee_detector.experiments.run_faruq_v3_af2_complement_audit',
    '--af2-checkpoint', str(AF2), '--output', str(STATIC), '--device', '0',
]
print('MENJALANKAN STATIC AUDIT:', ' '.join(command), flush=True)
with LOG.open('w', encoding='utf-8') as stream:
    completed = subprocess.run(command, cwd=REPO, stdout=stream, stderr=subprocess.STDOUT)
print('\n'.join(LOG.read_text(errors='replace').splitlines()[-200:]))
if completed.returncode != 0:
    if STATIC.is_file():
        print('STATIC AUDIT YANG SEMPAT TERSIMPAN:')
        print(STATIC.read_text(errors='replace'))
    raise RuntimeError(f'Static audit gagal: {completed.returncode}; log={LOG}')

audit = json.loads(STATIC.read_text())
parameters = {
    arm: {'total': row['parameters'], 'added': row['added_parameters']}
    for arm, row in audit['arms'].items()
}
initialization = {
    arm: {
        'input_exact': row['initial_af2_input_exact'],
        'output_bitwise_exact': row['initial_af2_output_exact'],
        'output_max_abs_diff': row['initial_af2_output_max_abs_diff'],
    }
    for arm, row in audit['arms'].items()
}
print('PARAMETERS:', json.dumps(parameters, indent=2))
print('INITIALIZATION:', json.dumps(initialization, indent=2))
print('GATES:', json.dumps(audit['gates'], indent=2))
print('DECISION:', audit['decision'])
print('SAVED:', STATIC)
assert audit['decision'] == 'PASS', 'STOP: static audit gagal; jangan training.'
print('PASS: empat arm boleh dijalankan terpisah/paralel. Test tetap terkunci.')